In [17]:
from rag_helper import RAGBase

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [18]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
#index = build_index(documents)
print(f"Loaded {len(documents)} documents")
# import json
# print(json.dumps(documents, indent=2))

Loaded 1380 documents


In [19]:
#Filter for the llm Zoomcamp docs
docs_llm = [doc for doc in documents if doc['course']=="llm-zoomcamp"]
print(f"LLM Zoomcamp has {len(docs_llm)} documents")

LLM Zoomcamp has 118 documents


In [20]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=['question','section','answer'],
    keyword_fields=['course'],
    id_field='doc_id',                 #---> use the existing id so if its re-run, it will update existing records instead of creating duplicates
    db_path='llm-faq.db'
)

#--> creates llm-faq.db in the directory where this script is run. If you want to store it somewhere else, change the db_path above.

In [22]:
import time

for doc in docs_llm:
    doc['doc_id'] = doc.pop("id")   # id is a  already a reserved column(keyword) in sqlite, need to add the ids from the json as  a different field name. 
    sqlite_index.add(doc)
    print(f"Added {doc['question'][:60]} ...")
    time.sleep(0.5)  # sleep for a bit to avoid overwhelming the database

Added I just discovered the course. Can I still join? ...
Added Course: I have registered for the LLM Zoomcamp. When can I e ...
Added What is the video/zoom link to the stream for the “Office Ho ...
Added How should I start the course and follow the weekly workflow ...
Added Leaderboard: I am not on the leaderboard / how do I know whi ...
Added Certificate: Can I follow the course in a self-paced mode an ...
Added I missed the first homework - can I still get a certificate? ...
Added Homework: Why does the content keep changing? ...
Added When will the course be offered next? ...
Added Are there any lectures/videos? Where are they? ...
Added Where can I track the LLM Zoomcamp syllabus, deadlines, home ...
Added Are there live sessions or office hours for each module? ...
Added Can I use Bluesky for learning in public credits? ...
Added Where is the LLM Zoomcamp Telegram channel? ...
Added Why doesn't the number of records I get in the FAQ dataset m ...
Added The homework submission fo

In [23]:
sqlite_index.close()

In [24]:
sqlite_index.search("how do i join the course")

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives).",
  'doc_id': '85384a18e5'},
 {'cours